# **Industrial Fleet TCO & Transition Analysis**

---



**Objective**

This project analyzes the 5-year Total Cost of Ownership (TCO) of electric and diesel commercial vehicles using operational and financial cost factors.

**Analysis Includes**
Purchase cost comparison
Fuel vs electricity cost analysis
Maintenance cost evaluation
Subsidy impact analysis
Resale value estimation
Fleet transition feasibility

**Workflow**
Excel Data
↓
SQL Structuring
↓
CSV Export
↓
Python Analysis
↓
Visualization & Insights

**Tools Used**
Python
Pandas
NumPy
Plotly
SQL Server
Excel
Tableau / Power BI

**Dataset**

The dataset was created using publicly available commercial vehicle specifications and simulated operational fleet assumptions for EV and diesel vehicles.

## 1. Setup and Imports

This section imports the Python libraries required for:

* data processing
* numerical calculations
* visualization
* analytical modeling

Pandas display settings are also configured for improved dataset readability during analysis.


In [6]:

# 1. Setup and Imports

import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings

import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.ensemble import RandomForestRegressor

# Configure Pandas display settings for improved readability
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)


## 2. Data Loading and Reshaping

This section loads the SQL-exported CSV dataset into Python using Pandas.

The dataset contains:

* vehicle specifications
* operational cost parameters
* business assumptions used for TCO analysis

Since the SQL export is stored in a long relational format, the data is r## 2. Data Loading and Reshaping

This section loads the SQL-exported CSV dataset into Python using Pandas.

The dataset includes:

* vehicle specifications
* operational cost parameters
* business assumptions used for TCO analysis

Since the SQL export is stored in a long relational format, the data is reshaped into:

* a structured vehicle DataFrame
* a separate assumptions dictionary for operational variables

This preprocessing step improves data organization and simplifies downstream analytical calculations and feature engineering.
eshaped into:

* a structured vehicle DataFrame
* a separate assumptions dictionary for operational variables

This improves data organization and simplifies downstream analytical calculations.


In [7]:

# 2. Data Loading and Reshaping

def load_and_reshape_sql_export(path):
    """
    Load the SQL-exported CSV dataset and reshape it into:
    - a structured vehicle DataFrame
    - an assumptions dictionary for operational variables
    """

    try:
        # Load SQL-exported CSV file
        df_long = pd.read_csv(path, header=None)

    except FileNotFoundError:
        raise FileNotFoundError(
            f"Dataset not found at: {path}"
        )

    # Assign column names to the SQL-exported dataset
    df_long.columns = [
        'vehicle_id',
        'vehicle_name',
        'ex_showroom_price',
        'charger_cost',
        'road_tax_reg_rate',
        'battery_capacity_kwh',
        'efficiency',
        'annual_maintenance_rate',
        'annual_tyre_cost',
        'resale_value_rate',
        'assumption_id',
        'assumption_name',
        'value'
    ]

    # -------------------------------
    # A. Create Vehicle DataFrame
    # -------------------------------

    vehicle_cols = [
        'vehicle_id',
        'vehicle_name',
        'ex_showroom_price',
        'charger_cost',
        'road_tax_reg_rate',
        'battery_capacity_kwh',
        'efficiency',
        'annual_maintenance_rate',
        'annual_tyre_cost',
        'resale_value_rate'
    ]

    # Extract unique vehicle records
    df_vehicles = (
        df_long[vehicle_cols]
        .drop_duplicates()
        .reset_index(drop=True)
    )

    # Classify vehicle type based on battery capacity
    df_vehicles['Vehicle_Type'] = np.where(
        df_vehicles['battery_capacity_kwh'] > 0,
        'EV',
        'Diesel'
    )

    # -------------------------------
    # B. Create Assumptions Dictionary
    # -------------------------------

    assumptions_df = (
        df_long[['assumption_name', 'value']]
        .drop_duplicates()
    )

    assumptions = (
        assumptions_df
        .set_index('assumption_name')['value']
        .to_dict()
    )

    print("Dataset successfully loaded and reshaped.")

    return df_vehicles, assumptions


# Dataset Path
DATA_PATH = '/content/TCO_Analysis_Diesel_vs_EV SQL.csv'

# Load processed dataset
df, assumptions = load_and_reshape_sql_export(DATA_PATH)


Dataset successfully loaded and reshaped.


In [8]:

# 1. Setup and Imports

import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings

import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.ensemble import RandomForestRegressor

# Configure Pandas display settings for improved readability
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

# ----------------------------------------
# Initial Dataset Inspection
# ----------------------------------------

# Load raw SQL-exported CSV dataset
df_preview = pd.read_csv(
    '/content/TCO_Analysis_Diesel_vs_EV SQL.csv',
    header=None
)

# Display first 10 rows of the dataset
df_preview.head(10)


,0,1,2,3,4,5,6,7,8,9,10,11,12
0,1,Diesel Tipper,5500000.0,0.0,0.08,0,1.5,0.040,160000.0,0.20,1,TCO_Period_Years,5.00
1,1,Diesel Tipper,5500000.0,0.0,0.08,0,1.5,0.040,160000.0,0.20,2,Annual_Distance_KM,45000.00
2,1,Diesel Tipper,5500000.0,0.0,0.08,0,1.5,0.040,160000.0,0.20,3,Diesel_Price_Per_Liter,98.66
3,1,Diesel Tipper,5500000.0,0.0,0.08,0,1.5,0.040,160000.0,0.20,4,Electricity_Price_Per_kWh,8.50
4,1,Diesel Tipper,5500000.0,0.0,0.08,0,1.5,0.040,160000.0,0.20,5,FAME_Subsidy_Per_kWh,10000.00
5,1,Diesel Tipper,5500000.0,0.0,0.08,0,1.5,0.040,160000.0,0.20,6,FAME_Subsidy_Cap_Rate,0.15
6,1,Diesel Tipper,5500000.0,0.0,0.08,0,1.5,0.040,160000.0,0.20,7,State_Subsidy_Cap,1000000.00
7,1,Diesel Tipper,5500000.0,0.0,0.08,0,1.5,0.040,160000.0,0.20,8,State_Subsidy_Rate,0.15
8,2,Propel 470 MEV,19000000.0,1000000.0,0.00,256,1.8,0.015,160000.0,0.25,1,TCO_Period_Years,5.00
9,2,Propel 470 MEV,19000000.0,1000000.0,0.00,256,1.8,0.015,160000.0,0.25,2,Annual_Distance_KM,45000.00


In [9]:
# Show unique vehicle names clearly
vehicle_names = df_preview[1].unique()

print("Total Unique Vehicles:", len(vehicle_names))
print("\nVehicle Names:\n")

for vehicle in vehicle_names:
    print(vehicle)

Total Unique Vehicles: 4

Vehicle Names:

Diesel Tipper
Propel 470 MEV
Olectra Meghaetron
Tata Prima E.28k


## 3. Feature Engineering: Cost Component Calculation

This section applies the business and financial logic required for the 5-year Total Cost of Ownership (TCO) analysis.

The preprocessing and feature engineering steps include:

* converting operational assumptions into numerical values
* calculating total vehicle acquisition cost
* estimating fuel and electricity operating costs
* computing annual maintenance expenses
* applying EV subsidy calculations
* estimating resale value after the operational lifecycle

These engineered features are used to support downstream TCO modeling, fleet comparison, and business decision analysis.


In [10]:
# 3. Feature Engineering: Create model-ready columns

print("\n--- Engineering features for TCO model ---")

# Ensure all assumption values are numeric for calculations
for key, value in assumptions.items():
    assumptions[key] = float(value)

# Calculate initial purchase price
df['Purchase_Price'] = df['ex_showroom_price'] + df['charger_cost']

# Calculate fuel cost per kilometer (different logic for Diesel vs. EV)
df['Fuel_Cost_per_km'] = np.where(
    df['Vehicle_Type'] == 'Diesel',
    assumptions['Diesel_Price_Per_Liter'] / df['efficiency'], # Diesel: Price per liter / km/liter
    assumptions['Electricity_Price_Per_kWh'] * df['efficiency'] # EV: Price per kWh * kWh/km
)

# Assign annual distance from assumptions
df['Annual_Km'] = assumptions['Annual_Distance_KM']

# Calculate annual maintenance cost
df['Maintenance_per_year'] = df['ex_showroom_price'] * df['annual_maintenance_rate']

# Calculate potential subsidies for EVs (0 for Diesel)
df['Subsidy'] = np.where(
    df['Vehicle_Type'] == 'EV',
    (df['ex_showroom_price'] * assumptions['FAME_Subsidy_Cap_Rate']) + assumptions['State_Subsidy_Cap'],
    0
)

# Calculate estimated resale value after 5 years
df['Resale_Value_after_5yrs'] = df['Purchase_Price'] * df['resale_value_rate']

# Battery replacement cost is set to 0 as per project scope (not explicitly provided in inputs)
df['Battery_Replacement_Cost'] = 0

print(" Feature engineering complete.")
print("\nPreview of key engineered features:")
display(df[['vehicle_name', 'Purchase_Price', 'Fuel_Cost_per_km', 'Maintenance_per_year', 'Subsidy']].head())


--- Engineering features for TCO model ---
 Feature engineering complete.

Preview of key engineered features:


,vehicle_name,Purchase_Price,Fuel_Cost_per_km,Maintenance_per_year,Subsidy
0,Diesel Tipper,5500000.0,65.773333,220000.0,0.0
1,Propel 470 MEV,20000000.0,15.300000,285000.0,3850000.0
2,Olectra Meghaetron,16500000.0,13.600000,232500.0,3325000.0
3,Tata Prima E.28k,17000000.0,14.025000,240000.0,3400000.0


## 4. Calculating 5-Year Total Cost of Ownership (TCO)

This section calculates the 5-year Total Cost of Ownership (TCO) for each commercial vehicle using the engineered operational and financial features.

The TCO model consolidates:

* acquisition cost
* operational fuel/electricity expenses
* maintenance and tyre costs
* government subsidies
* lifecycle resale value

The objective is to estimate the long-term financial impact of EV and diesel fleet operations and support comparative fleet transition analysis.


In [11]:
# 4. TCO Calculation

def compute_5yr_tco(row):
    """
    Calculates the 5-year Total Cost of Ownership for a given vehicle row.
    """
    total_fuel_cost_5yrs = row['Fuel_Cost_per_km'] * row['Annual_Km'] * 5
    total_maintenance_cost_5yrs = (row['Maintenance_per_year'] + row['annual_tyre_cost']) * 5

    # Gross cost includes purchase, fuel, maintenance, and battery replacement
    gross_cost = row['Purchase_Price'] - row['Subsidy'] + total_fuel_cost_5yrs + total_maintenance_cost_5yrs + row['Battery_Replacement_Cost']

    # Net TCO is gross cost minus the resale value (which acts as a credit)
    net_tco = gross_cost - row['Resale_Value_after_5yrs']
    return net_tco

# Apply the TCO calculation function to each row of the DataFrame
df['TCO_5yr'] = df.apply(compute_5yr_tco, axis=1)

print('\n--- 5-Year TCO computed successfully ---')
print("Vehicles ranked by 5-Year TCO:")
display(df[['vehicle_name', 'Purchase_Price', 'TCO_5yr']].sort_values('TCO_5yr'))


--- 5-Year TCO computed successfully ---
Vehicles ranked by 5-Year TCO:


,vehicle_name,Purchase_Price,TCO_5yr
2,Olectra Meghaetron,16500000.0,14072500.0
3,Tata Prima E.28k,17000000.0,14505625.0
1,Propel 470 MEV,20000000.0,16817500.0
0,Diesel Tipper,5500000.0,21099000.0


## 5. Visualizing TCO Distribution: EV vs Diesel

This section visualizes the distribution of 5-year Total Cost of Ownership (TCO) between EV and diesel commercial vehicles.

An interactive violin plot is used to compare:

* TCO distribution
* cost density
* median ownership cost
* variation between vehicle categories

This visualization supports comparative fleet transition analysis by highlighting long-term operational and financial differences between EV and diesel fleets.


In [12]:
# 5. Visualizations: TCO Distribution

print("\n--- Comparative Summary: EV vs Diesel ---")
# Aggregate summary statistics for TCO by Vehicle Type
comp_summary = df.groupby('Vehicle_Type').agg(
    Number_of_Vehicles=('TCO_5yr','size'),
    Mean_TCO_5yr=('TCO_5yr','mean'),
    Median_TCO_5yr=('TCO_5yr','median'),
    Mean_Purchase_Price=('Purchase_Price','mean'),
    Mean_Resale_Value=('Resale_Value_after_5yrs','mean')
).reset_index()
display(comp_summary)

# Create an interactive Violin Plot
fig_violin = px.violin(df, x='Vehicle_Type', y='TCO_5yr', color='Vehicle_Type',
                box=True, points='all', hover_data=['vehicle_name', 'Purchase_Price'],
                title='<b>TCO 5-year Distribution: EV vs Diesel (from SQL Data)</b>',
                labels={'TCO_5yr': '5-Year Total Cost of Ownership (INR)', 'Vehicle_Type': ''})
fig_violin.update_layout(showlegend=False) # Hide legend as color is redundant with x-axis
fig_violin.show()


--- Comparative Summary: EV vs Diesel ---


,Vehicle_Type,Number_of_Vehicles,Mean_TCO_5yr,Median_TCO_5yr,Mean_Purchase_Price,Mean_Resale_Value
0,Diesel,1,21099000.0,21099000.0,5.500000e+06,1.100000e+06
1,EV,3,15131875.0,14505625.0,1.783333e+07,4.458333e+06


## **6. Visualizing TCO Cost Component Breakdown**

This stacked bar chart breaks down the total 5-year TCO for each individual vehicle into its contributing components: initial net cost (purchase - subsidy), 5-year fuel, 5-year maintenance/tyre, and resale credit. This helps in understanding where the costs are allocated.

In [ ]:
# 6. Visualizations: Cost Component Breakdown

# Calculate 5-year totals for each cost component
df['Initial_Cost_Net'] = df['Purchase_Price'] - df['Subsidy']
df['Fuel_5yr'] = df['Fuel_Cost_per_km'] * df['Annual_Km'] * 5
df['Maint_Tyre_5yr'] = (df['Maintenance_per_year'] + df['annual_tyre_cost']) * 5
df['Resale_Credit'] = -1 * df['Resale_Value_after_5yrs'] # Represent resale as a negative cost (credit)

# Melt the DataFrame to prepare for stacked bar chart (long format needed)
comp_cols = ['vehicle_name', 'Initial_Cost_Net', 'Fuel_5yr', 'Maint_Tyre_5yr', 'Resale_Credit']
comp_df = df[comp_cols].melt(id_vars='vehicle_name', var_name='Cost_Component', value_name='Amount')

# Create an interactive Stacked Bar Chart
fig_bar_components = px.bar(comp_df, x='vehicle_name', y='Amount', color='Cost_Component',
             title='<b>5-Year TCO Cost Component Breakdown (from SQL Data)</b>',
             labels={'Amount': 'Cost/Credit Amount (INR)', 'vehicle_name': 'Vehicle'},
             category_orders={'vehicle_name': df.sort_values('TCO_5yr')['vehicle_name'].tolist()}) # Order by total TCO
fig_bar_components.show()

## 7. Identifying Key Drivers of TCO (Feature Importance)

This section uses a Random Forest Regressor as an exploratory analytical technique to identify the most influential variables affecting 5-year Total Cost of Ownership (TCO).

The objective is not predictive modeling, but to evaluate the relative contribution of operational and financial factors such as:

* vehicle acquisition cost
* fuel/electricity operating cost
* maintenance expense
* tyre cost
* resale value
* vehicle efficiency

Feature importance analysis helps identify the dominant cost drivers influencing long-term fleet economics and supports data-driven operational decision-making.


In [13]:
# 7. Feature Importance for TCO (Random Forest - for insights)

# Define the input features that influence TCO
feature_cols = [
    'ex_showroom_price', 'charger_cost', 'efficiency',
    'annual_maintenance_rate', 'annual_tyre_cost', 'resale_value_rate',
    'Fuel_Cost_per_km' # Our engineered feature, which is a blend of other assumptions
]
# Filter out any columns that might not exist in the DataFrame for robustness
feature_cols = [col for col in feature_cols if col in df.columns]

X = df[feature_cols] # Features (input variables)
y = df['TCO_5yr']     # Target (what we want to explain)

# Ensure there's enough data points for Random Forest (needs at least 2 samples)
if len(X) > 1 and len(y) > 1:
    # Initialize and train a Random Forest Regressor
    rf = RandomForestRegressor(n_estimators=100, random_state=42) # 100 trees, fixed random state for reproducibility
    rf.fit(X, y)

    # Extract feature importances
    imp_df = pd.DataFrame({'feature': X.columns, 'importance': rf.feature_importances_}).sort_values('importance', ascending=False)

    print('\n--- Top Features Influencing 5-Year TCO (from SQL Data) ---')
    display(imp_df)

    # Create an interactive Horizontal Bar Chart for Feature Importance
    fig_importance = px.bar(imp_df, x='importance', y='feature', orientation='h',
                 title='<b>Key Drivers of 5-Year TCO (from SQL Data)</b>',
                 labels={'importance': 'Feature Importance Score', 'feature': 'Input Factor'})
    fig_importance.update_layout(yaxis={'categoryorder':'total ascending'}) # Order bars from lowest to highest importance
    fig_importance.show()
else:
    print("\n⚠️ Not enough data points to compute meaningful Feature Importance (requires at least 2 samples).")


--- Top Features Influencing 5-Year TCO (from SQL Data) ---


,feature,importance
2,efficiency,0.235214
6,Fuel_Cost_per_km,0.220395
0,ex_showroom_price,0.211931
1,charger_cost,0.132526
5,resale_value_rate,0.122872
3,annual_maintenance_rate,0.077062
4,annual_tyre_cost,0.000000


## 8. Key Findings and Business Insights

Based on the operational analysis and TCO modeling performed in this project, the following business insights were identified:

---

### EV Fleets Demonstrate Lower Long-Term Operational Cost

The analysis indicates that EV commercial vehicles achieve significantly lower operational energy costs compared to diesel fleets, resulting in improved long-term Total Cost of Ownership (TCO).

---

### Fuel Cost per Kilometer is a Major Cost Driver

Feature importance analysis identified Fuel_Cost_per_km as one of the most influential variables affecting long-term fleet ownership cost.

Diesel fleets showed substantially higher lifecycle fuel expenses due to higher fuel cost and lower operational efficiency.

---

### Higher Initial Investment vs Long-Term Savings

EV fleets require higher upfront acquisition investment because of battery systems and charging infrastructure.

However, lower operational energy expenses and reduced lifecycle operating costs improve long-term financial feasibility.

---

### Government Subsidies Improve EV Adoption Feasibility

Government subsidy programs significantly reduce effective EV acquisition cost and help accelerate breakeven periods for fleet transition.

---

### Lifecycle Resale Value Reduces Net Ownership Cost

Residual resale value contributes to lowering final lifecycle ownership cost and improves long-term asset utilization economics.

---

### Business Recommendation

For high-mileage commercial fleet operations, EV adoption demonstrates strong long-term financial potential due to lower operational energy expenses and improved lifecycle cost efficiency.

Fleet transition feasibility improves further when government incentives and long-term operational savings are incorporated into strategic investment planning.
